In [0]:
from pyspark.sql.functions import col
# 1. Identify a default dataset: 'samples.nyctaxi.trips' is available in Unity Catalog
source_table = "samples.nyctaxi.trips"

# 2. Consume the data using structured streaming
streaming_df = spark.readStream.table(source_table)

# 3. Write output files to a directory in Delta format
output_path = "/Volumes/workspace/default/my_data_volume/nyctaxi_stream_output"
checkpoint_path_10001 = "/Volumes/workspace/default/my_data_volume/nyctaxi_stream_checkpoint_10001"
checkpoint_path_10044 = "/Volumes/workspace/default/my_data_volume/nyctaxi_stream_checkpoint_10044"

streaming_query_10001 = streaming_df.filter(col("pickup_zip")=='10001').writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_path_10001) \
    .trigger(availableNow=True) \
    .start(output_path)

streaming_query_10044 = streaming_df.filter(col("pickup_zip")=='10044').writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_path_10044) \
    .trigger(availableNow=True) \
    .start(output_path)

In [0]:
df = spark.read.format("delta").load("/Volumes/workspace/default/my_data_volume/nyctaxi_stream_output")
display(df.groupBy("pickup_zip").count())


In [0]:
# stop streaming
for stream in spark.streams.active:
  # stream.stop()
  # stream.awaitTermination()
  print(stream)

In [0]:
streaming_df

In [0]:
streaming_query_10044.status

In [0]:
%sql
DESCRIBE HISTORY '/Volumes/workspace/default/my_data_volume/nyctaxi_stream_output'
